In [3]:
%pip install -q sentence-transformers faiss-cpu chromadb transformers pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os, platform, time
import numpy as np
import pandas as pd

# if platform.system() == "Darwin":          # Mac only; Colab ignores this
#     os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import nltk, string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for r in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(r, quiet=True)

from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

print("Setup complete!")

Setup complete!


In [5]:
REVIEWS = [
    "At McDonald's the food was ok and the service was bad.",
    "I wouldn't recommend this Japanese restaurant to anyone.",
    "I loved this restaurant when I traveled to Thailand last summer.",
    "The menu at Loving Hut has a wide variety of vegan options.",
    "The staff was friendly and helpful at Google's employee cafeteria.",
    "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
    "I had a terrible experience at Pizza Hut. The pizza was burnt and the service was slow.",
    "The sushi at Sushi Express is always fresh and the prices are reasonable.",
    "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
    "The dessert selection at Sweet Treats is amazing and the prices are fair.",
]
PARAPHRASE = "somewhere with good raw fish that is not expensive"

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t not in string.punctuation and t not in stop_words]
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens)

clean_reviews = [preprocess_text(r) for r in REVIEWS]
tfidf = TfidfVectorizer().fit(clean_reviews)
doc_vectors = tfidf.transform(clean_reviews)

def keyword_search(q):
    return cosine_similarity(tfidf.transform([preprocess_text(q)]), doc_vectors)[0]

kw = keyword_search(PARAPHRASE)
print(f"Yesterday's best score on '{PARAPHRASE}': {kw.max():.3f}")

Yesterday's best score on 'somewhere with good raw fish that is not expensive': 0.000


In [6]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
review_vectors = embedder.encode(REVIEWS, normalize_embeddings=True)

print(f"review_vectors: {review_vectors.shape}   <- 10 reviews, 384 numbers each")

q = embedder.encode([PARAPHRASE], normalize_embeddings=True)
meaning = (review_vectors @ q.T).ravel()

print(f"\nQUESTION: '{PARAPHRASE}'\n")
print(f"{'YESTERDAY':>10} {'TODAY':>7}   review")
for i in np.argsort(meaning)[::-1][:5]:
    print(f"{kw[i]:>10.3f} {meaning[i]:>7.3f}   [{i}] {REVIEWS[i][:52]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2660.89it/s]


review_vectors: (10, 384)   <- 10 reviews, 384 numbers each

QUESTION: 'somewhere with good raw fish that is not expensive'

 YESTERDAY   TODAY   review
     0.000   0.425   [7] The sushi at Sushi Express is always fresh and the p
     0.000   0.332   [9] The dessert selection at Sweet Treats is amazing and
     0.000   0.329   [1] I wouldn't recommend this Japanese restaurant to any
     0.000   0.319   [3] The menu at Loving Hut has a wide variety of vegan o
     0.000   0.302   [2] I loved this restaurant when I traveled to Thailand 


In [7]:
TICKETS = [
    "Invoice INV-2024-8871 was paid in full on March 3rd.",
    "Invoice INV-2024-8817 is overdue by 45 days and has been sent to collections.",
    "Invoice INV-2024-1788 was cancelled at the customer's request.",
    "Invoice INV-2023-8817 was refunded after a duplicate charge.",
    "The customer called to ask about an unpaid invoice from last spring.",
]
LOOKUP = "INV-2024-8817"

tv = embedder.encode(TICKETS, normalize_embeddings=True)
dense = (tv @ embedder.encode([LOOKUP], normalize_embeddings=True).T).ravel()

tt = TfidfVectorizer().fit(TICKETS)
kws = cosine_similarity(tt.transform([LOOKUP]), tt.transform(TICKETS))[0]

print(f"LOOKING FOR: {LOOKUP}\n")
print("MEANING says:")
for i in np.argsort(dense)[::-1][:3]:
    print(f"   {dense[i]:.3f}  {TICKETS[i][:58]}")
print("\nKEYWORDS say:")
for i in np.argsort(kws)[::-1][:3]:
    print(f"   {kws[i]:.3f}  {TICKETS[i][:58]}")

LOOKING FOR: INV-2024-8817

MEANING says:
   0.515  Invoice INV-2024-8871 was paid in full on March 3rd.
   0.492  Invoice INV-2024-8817 is overdue by 45 days and has been s
   0.479  Invoice INV-2023-8817 was refunded after a duplicate charg

KEYWORDS say:
   0.339  Invoice INV-2024-8817 is overdue by 45 days and has been s
   0.316  Invoice INV-2023-8817 was refunded after a duplicate charg
   0.248  Invoice INV-2024-1788 was cancelled at the customer's requ


In [8]:
def rank_of(scores):
    """Position of every document in a ranking: 1 = best."""
    order = np.argsort(scores)[::-1]
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks

K = 60   # the standard constant; it damps the weight of low positions
rrf = 1 / (K + rank_of(dense)) + 1 / (K + rank_of(kws))

print(f"{'MEANING':>8} {'KEYWORD':>8} {'HYBRID':>7}   document")
print("=" * 88)
for i in np.argsort(rrf)[::-1]:
    print(f"{rank_of(dense)[i]:>8} {rank_of(kws)[i]:>8} {rank_of(rrf)[i]:>7}   {TICKETS[i][:58]}")

print(f"\n-> Hybrid top hit: {TICKETS[int(np.argmax(rrf))]}")

 MEANING  KEYWORD  HYBRID   document
       2        1       1   Invoice INV-2024-8817 is overdue by 45 days and has been s
       1        4       2   Invoice INV-2024-8871 was paid in full on March 3rd.
       3        2       3   Invoice INV-2023-8817 was refunded after a duplicate charg
       4        3       4   Invoice INV-2024-1788 was cancelled at the customer's requ
       5        5       5   The customer called to ask about an unpaid invoice from la

-> Hybrid top hit: Invoice INV-2024-8817 is overdue by 45 days and has been sent to collections.


In [9]:
AG_NEWS = ("https://huggingface.co/datasets/fancyzhx/ag_news/"
           "resolve/main/data/train-00000-of-00001.parquet")
TOPICS = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

news = pd.read_parquet(AG_NEWS).head(10_000).copy()
news["topic"] = news["label"].map(TOPICS)
news = news.reset_index(drop=True)
news["id"] = news.index

start = time.time()
news_vectors = embedder.encode(news["text"].tolist(), normalize_embeddings=True,
                               batch_size=256, show_progress_bar=True).astype('float32')
print(f"\nEncoded {len(news):,} articles in {time.time()-start:.1f} seconds")
print(f"news_vectors: {news_vectors.shape}")

Batches: 100%|██████████| 40/40 [01:59<00:00,  2.98s/it]


Encoded 10,000 articles in 119.3 seconds
news_vectors: (10000, 384)


In [10]:
def knn_search(question, k=5):
    """Exact search: compare the question against every article."""
    q = embedder.encode([question], normalize_embeddings=True).astype('float32')
    scores = (news_vectors @ q.T).ravel()
    top = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top]

QUESTION = "How many medals did Michael Phelps win at the Athens Olympics?"
q_vec = embedder.encode([QUESTION], normalize_embeddings=True).astype('float32')

runs = 20
start = time.time()
for _ in range(runs):
    _ = news_vectors @ q_vec.T
per_query_ms = (time.time() - start) / runs * 1000

print(f"{'documents':>14}  {'time per question':>18}  {'memory for the vectors':>24}")
print("=" * 62)
for n in [len(news), 1_000_000, 10_000_000, 100_000_000]:
    ms = per_query_ms * n / len(news)
    gb = n * 384 * 4 / 1e9
    measured = "   <- measured" if n == len(news) else ""
    shown = f"{ms:,.1f} ms" if ms < 1000 else f"{ms/1000:,.1f} s"
    ram = f"{gb*1000:,.0f} MB" if gb < 1 else f"{gb:,.1f} GB"
    print(f"{n:>14,}  {shown:>18}  {ram:>24}{measured}")

print("\nAnd the answers are exact — it looked at every article:")
for i, sc in knn_search(QUESTION, k=3):
    print(f"   {sc:.3f}  {news.text[i][:88]}")

     documents   time per question    memory for the vectors
        10,000              0.9 ms                     15 MB   <- measured
     1,000,000             85.4 ms                    1.5 GB
    10,000,000            854.4 ms                   15.4 GB
   100,000,000               8.5 s                  153.6 GB

And the answers are exact — it looked at every article:
   0.808  Phelps to Forgo Final Race of Olympics ATHENS, Greece - Michael Phelps is done for the O
   0.801  Swimming: Phelps Wins Eighth Medal at Athens Games  ATHENS (Reuters) - American swimmer 
   0.800  Phelps wins fourth gold at Athens Olympics ATHENS, Aug. 19 (Xinhuanet) -- Michael Phelps


In [11]:
QUESTION = "How many medals did Michael Phelps win at the Athens Olympics?"

index = faiss.IndexIDMap(faiss.IndexFlatIP(384))
index.add_with_ids(news_vectors, news["id"].to_numpy().astype('int64'))
print(f"Index built: {index.ntotal:,} vectors\n")

def faiss_search(question, k=5):
    q = embedder.encode([question], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

for i, s in faiss_search(QUESTION, k=3):
    print(f"   {s:.3f}  [{i}] {news.text[i][:72]}")

Index built: 10,000 vectors

   0.808  [5308] Phelps to Forgo Final Race of Olympics ATHENS, Greece - Michael Phelps i
   0.801  [5811] Swimming: Phelps Wins Eighth Medal at Athens Games  ATHENS (Reuters) - A
   0.800  [4506] Phelps wins fourth gold at Athens Olympics ATHENS, Aug. 19 (Xinhuanet) -


In [12]:
PROBE_QUESTIONS = [          # measure on several questions, not one
    "How many gold medals did Michael Phelps win at the Athens Olympics?",
    "oil prices and the economy", "new Microsoft software release",
    "election results in Europe", "stock market falls",
    "space telescope discovery", "Apple computer sales",
    "football transfer news", "hurricane damage in Florida",
    "peace talks in the Middle East", "internet security flaw", "airline bankruptcy",
]
probe_vectors = embedder.encode(PROBE_QUESTIONS, normalize_embeddings=True).astype('float32')

def timed_search(idx, k=10, runs=20):
    """Average milliseconds per question, and the ids each question got back."""
    start = time.time()
    for _ in range(runs):
        _, ids = idx.search(probe_vectors, k)
    ms = (time.time() - start) / runs / len(PROBE_QUESTIONS) * 1000
    return ms, ids

flat = faiss.IndexFlatIP(384)
flat.add(news_vectors)
exact_ms, exact_ids = timed_search(flat)      # the answer key

NLIST = 200
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(384), 384, NLIST, faiss.METRIC_INNER_PRODUCT)
ivf.train(news_vectors)                       # the clustering step — only IVF needs this
ivf.add(news_vectors)

print(f"{'nprobe':>7}  {'clusters read':>14}  {'ms per question':>16}  {'recall@10':>10}")
print("=" * 56)
for nprobe in [1, 2, 5, 10, 25, NLIST]:
    ivf.nprobe = nprobe
    ms, ids = timed_search(ivf)
    recall = np.mean([len(set(ids[j]) & set(exact_ids[j])) / 10
                      for j in range(len(PROBE_QUESTIONS))])
    tail = "   <- this IS exact search" if nprobe == NLIST else ""
    print(f"{nprobe:>7}  {nprobe/NLIST:>13.1%}  {ms:>16.3f}  {recall:>10.2f}{tail}")

print(f"\nexact search: {exact_ms:.3f} ms per question, recall 1.00 by definition")

 nprobe   clusters read   ms per question   recall@10
      1           0.5%             0.010        0.70
      2           1.0%             0.008        0.83
      5           2.5%             0.021        0.95
     10           5.0%             0.054        0.98
     25          12.5%             0.050        1.00
    200         100.0%             0.384        1.00   <- this IS exact search

exact search: 0.229 ms per question, recall 1.00 by definition


In [13]:
hnsw = faiss.IndexHNSWFlat(384, 32, faiss.METRIC_INNER_PRODUCT)  # 32 = links per vector
start = time.time()
hnsw.add(news_vectors)
build_time = time.time() - start

print(f"{'efSearch':>9}  {'ms per question':>16}  {'recall@10':>10}")
print("=" * 40)
for ef in [16, 32, 64]:
    hnsw.hnsw.efSearch = ef                  # how widely to walk the graph
    ms, ids = timed_search(hnsw)
    recall = np.mean([len(set(ids[j]) & set(exact_ids[j])) / 10
                      for j in range(len(PROBE_QUESTIONS))])
    print(f"{ef:>9}  {ms:>16.3f}  {recall:>10.2f}")

print(f"\nBuilt in {build_time:.1f}s. Uses more memory than IVF — it stores the graph too.")

 efSearch   ms per question   recall@10
       16             0.013        1.00
       32             0.021        1.00
       64             0.071        1.00

Built in 0.4s. Uses more memory than IVF — it stores the graph too.


In [14]:
scores, ids = index.search(q_vec, 5)
print("Everything FAISS returns:")
print(f"   ids    = {ids[0].tolist()}")
print(f"   scores = {[round(float(x), 3) for x in scores[0]]}")
print("\n-> No text. No topic. Those live in `news`, and we join them by hand.\n")

print("QUESTION WE CANNOT ASK: 'articles about apple, but only Business ones'")
print("=" * 78)
apple = faiss_search("apple", k=5)
kept = 0
for i, sc in apple:
    topic = news.topic[i]
    mark = "keep" if topic == "Business" else "drop"
    kept += topic == "Business"
    print(f"   [{mark}] {topic:9} {sc:.3f}  {news.text[i][:60]}")
print(f"\n-> Asked for 5, kept {kept}.")

Everything FAISS returns:
   ids    = [5308, 5811, 4506, 4096, 6687]
   scores = [0.808, 0.801, 0.8, 0.793, 0.791]

-> No text. No topic. Those live in `news`, and we join them by hand.

QUESTION WE CANNOT ASK: 'articles about apple, but only Business ones'
   [drop] Sci/Tech  0.605  Rival Targets Apple's ITunes Music Customers By ALEX VEIGA  
   [drop] Sci/Tech  0.602  iPod is the only clear winner? While Apple is cruising along
   [drop] Sci/Tech  0.564  Apple G5 chips  #39;in short supply #39; Apple may be facing
   [drop] Sci/Tech  0.528  Apple in iTunes.co.uk cybersquatting row Apple has been accu
   [drop] Sci/Tech  0.527  Satisfied Customers Jefferson Graham writes for USA Today,  

-> Asked for 5, kept 0.


In [15]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()
try:
    client.delete_collection("news-collection")     # so re-running is safe
except Exception:
    pass

collection = client.create_collection(
    name="news-collection",
    embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"),
    metadata={"hnsw:space": "cosine"},
)

for s in range(0, len(news), 4000):                 # add in batches - there is a limit
    part = news.iloc[s:s+4000]
    collection.add(ids=[str(i) for i in part["id"]],
                   documents=part["text"].tolist(),
                   metadatas=[{"topic": t} for t in part["topic"]])

def show(res, title):
    print(title)
    for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"   dist={dist:.3f}  [{m['topic']:8}] {d[:60]}")
    print()

show(collection.query(query_texts=["apple"], n_results=3), "'apple', no filter:")
show(collection.query(query_texts=["apple"], n_results=3,
                      where={"topic": "Business"}), "'apple', Business only:")

'apple', no filter:
   dist=0.395  [Sci/Tech] Rival Targets Apple's ITunes Music Customers By ALEX VEIGA  
   dist=0.398  [Sci/Tech] iPod is the only clear winner? While Apple is cruising along
   dist=0.436  [Sci/Tech] Apple G5 chips  #39;in short supply #39; Apple may be facing

'apple', Business only:
   dist=0.508  [Business] Dutch Firm Beats Apple to Punch A music retailer from the Ne
   dist=0.529  [Business] Site Is Mac Daddy of Apple Deals A site for cut-rate Apple p
   dist=0.608  [Business] New Device: Flying Robot Seiko Epson hopes the tiny robot wi



In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
lm = AutoModelForCausalLM.from_pretrained(MODEL_ID)
generator = pipeline("text-generation", model=lm, tokenizer=tok)

def ask(messages, max_new_tokens=60):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = generator(prompt, max_new_tokens=max_new_tokens,
                    do_sample=False, return_full_text=False)
    return out[0]["generated_text"].strip()

print(f"QUESTION: {QUESTION}\n")
print("ANSWER, with no documents:")
print("  ", ask([{"role": "user", "content": QUESTION}]))

c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avigd\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `

QUESTION: How many medals did Michael Phelps win at the Athens Olympics?

ANSWER, with no documents:


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


   Michael Phelps won 24 gold medals at the 2008 Summer Olympics in Athens, Greece. He also won 15 silver and 13 bronze medals. This made him the most decorated Olympian of all time with 47 total medals across all sports. His record-breaking


In [18]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

candidates = faiss_search(QUESTION, k=20)                      # stage 1: fast, broad
pairs = [(QUESTION, news.text[i]) for i, _ in candidates]
rerank_scores = reranker.predict(pairs)                        # stage 2: slow, accurate

before = [i for i, _ in candidates][:3]
after  = [candidates[j][0] for j in np.argsort(rerank_scores)[::-1]][:3]

print(f"Top 3 straight from the search : {answer_from(before)}")
print(f"Top 3 after reranking          : {answer_from(after)}")

c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avigd\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4762.23it/s]
[transformers] Both `max_ne

Top 3 straight from the search : fourth gold
Top 3 after reranking          : six
